In [1]:
import pandas as pd
import numpy as np
from scipy import stats

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/cleaned/fraud_features.csv")
print(df.shape)   

(15419, 50)


In [2]:
ct = pd.crosstab(df["PolicyType"], df["FraudFound_P"])

chi2, p_value, dof, expected = stats.chi2_contingency(ct)

n = ct.sum().sum()
k = min(ct.shape) - 1
cramers_v = np.sqrt(chi2 / (n * k))

print(f"Chi-square: {chi2:.2f}, dof: {dof}, p-value: {p_value:.2e}, Cramér's V: {cramers_v:.3f}")

Chi-square: 437.49, dof: 8, p-value: 1.77e-89, Cramér's V: 0.168


In [3]:
# Fault alone
ct_fault = pd.crosstab(df["Fault"], df["FraudFound_P"])
chi2_f, p_f, dof_f, _ = stats.chi2_contingency(ct_fault)
v_f = np.sqrt(chi2_f / (ct_fault.sum().sum() * (min(ct_fault.shape)-1)))
print(f"Fault: chi2={chi2_f:.2f}, p={p_f:.2e}, Cramér's V={v_f:.3f}")

# Police Report alone
ct_police = pd.crosstab(df["PoliceReportFiled"], df["FraudFound_P"])
chi2_p, p_p, dof_p, _ = stats.chi2_contingency(ct_police)
v_p = np.sqrt(chi2_p / (ct_police.sum().sum() * (min(ct_police.shape)-1)))
print(f"PoliceReportFiled: chi2={chi2_p:.2f}, p={p_p:.2e}, Cramér's V={v_p:.3f}")

Fault: chi2=264.98, p=1.41e-59, Cramér's V=0.131
PoliceReportFiled: chi2=3.55, p=5.95e-02, Cramér's V=0.015


In [4]:
corr, p_value = stats.spearmanr(df["PastNumberOfClaims_ord"], df["FraudFound_P"])
print(f"Spearman correlation (PastNumberOfClaims_ord vs FraudFound_P): {corr:.3f}, p-value: {p_value:.2e}")

Spearman correlation (PastNumberOfClaims_ord vs FraudFound_P): -0.058, p-value: 7.61e-13


In [5]:
# Collapse to: "recent change" (under 6 months + 2 to 3 years) vs "no/older change"
df["Recent_Address_Change"] = df["AddressChange_Claim"].isin(["under 6 months", "2 to 3 years"]).astype(int)

ct_addr = pd.crosstab(df["Recent_Address_Change"], df["FraudFound_P"])
print(ct_addr)

odds_ratio, p_value = stats.fisher_exact(ct_addr)
print(f"Fisher's Exact Test — Odds Ratio: {odds_ratio:.2f}, p-value: {p_value:.2e}")

FraudFound_P               0    1
Recent_Address_Change            
0                      14255  869
1                        241   54
Fisher's Exact Test — Odds Ratio: 3.68, p-value: 1.18e-13


In [6]:
p_values = {
    "H3 Policy Type (chi2)": 1.77e-89,
    "H4 Fault (chi2)": 1.41e-59,
    "H4 Police Report (chi2)": 5.95e-02,
    "H5 Past Claims trend (Spearman)": 7.61e-13,
    "Address Change (Fisher's Exact)": 1.18e-13,
}

alpha = 0.05
bonferroni_alpha = alpha / len(p_values)
print(f"Bonferroni-corrected significance threshold: {bonferroni_alpha:.5f}")
print()
for name, p in p_values.items():
    sig = "significant" if p < bonferroni_alpha else "not significant"
    print(f"{name}: p = {p:.2e} → {sig}")

Bonferroni-corrected significance threshold: 0.01000

H3 Policy Type (chi2): p = 1.77e-89 → significant
H4 Fault (chi2): p = 1.41e-59 → significant
H4 Police Report (chi2): p = 5.95e-02 → not significant
H5 Past Claims trend (Spearman): p = 7.61e-13 → significant
Address Change (Fisher's Exact): p = 1.18e-13 → significant


## Statistical Test Results

| Hypothesis | Test | Statistic | p-value | Effect size | Verdict |
|---|---|---|---|---|---|
| H3: Policy Type → Fraud | Chi-square | χ²(8) = 437.49 | 1.77e-89 | Cramér's V = 0.168 | Confirmed |
| H4: Fault → Fraud | Chi-square | χ² = 264.98 | 1.41e-59 | Cramér's V = 0.131 | Confirmed |
| H4: Police Report → Fraud | Chi-square | χ² = 3.55 | 5.95e-02 | Cramér's V = 0.015 | **Not significant alone** — effect in SQL was likely confounded with Fault |
| H5: Past Claims trend | Spearman | ρ = -0.058 | 7.61e-13 | small negative | Confirmed (rejects original hypothesis — more history = lower fraud) |
| Address Change → Fraud | Fisher's Exact | OR = 3.68 | 1.18e-13 | ~3.7x odds | Confirmed |

**Multiple testing correction:** Bonferroni-adjusted α = 0.01. Four of five results remain significant;
Police Report alone does not clear even the uncorrected α = 0.05 bar, showing its apparent effect in
Step 4's SQL cross-tab was likely driven by its association with Fault, not an independent signal.

**Why Fisher's Exact Test for Address Change:** the original "under 6 months" bucket had only 4
observations, too few for chi-square's assumptions. Grouping it with "2 to 3 years" and using
Fisher's Exact Test (appropriate for small samples) gives a reliable odds ratio of 3.68.